# Demostración de FET (Free Timetabling Software) con datos reales de oferta académica

Este notebook funciona tanto en **Google Colab** como en un Jupyter local:

1. **Descarga** el mirror de FET (`github.com/nico-alvz/fet-mirror`) y el binario `fet-cl` (línea de comandos).
2. **Carga un archivo de Oferta Académica** (`.xlsx`) — si no lo encuentra, te lo pide (en Colab y Binder, con un selector de subida: los datos no se guardan en el repositorio).
3. **Construye** un problema de timetabling para FET a partir de esos datos (asignaturas, docentes, secciones/grupos y salas reales del archivo), **inventando únicamente la capacidad de las salas** (dato que la oferta académica no incluye).
4. **Ejecuta FET** para que genere un horario desde cero, respetando disponibilidad de docentes, grupos y salas.
5. **Verifica y visualiza** el horario resultante, comprobando que no existan choques de horario.

> Nota: la oferta académica original ya trae un horario asignado (día/bloque/sala reales). Para demostrar el *funcionamiento* de FET como motor de generación de horarios, se toman de ese archivo las entidades del problema (asignaturas, docentes, grupos, salas) pero **no** los horarios ya asignados — esos los calcula FET.


In [ ]:
import os, subprocess, glob, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_DIR = "/content" if IN_COLAB else os.path.abspath(os.getcwd())
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

REPO_DIR = os.path.join(PROJECT_DIR, "fet-mirror")
BIN_DIR = os.path.join(PROJECT_DIR, "fet-bin")
FET_CL = os.path.join(BIN_DIR, "fet-cl-7.10.5", "bin", "fet-cl")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/nico-alvz/fet-mirror.git", REPO_DIR], check=True)
else:
    print("fet-mirror ya está clonado en", REPO_DIR)

if not os.path.isfile(FET_CL):
    # Se usa el build "command-line-only" (sin las librerias de GUI de Qt), mucho
    # mas liviano que el bundle completo de fet, ya que el notebook solo necesita fet-cl.
    os.makedirs(BIN_DIR, exist_ok=True)
    tar_path = os.path.join(BIN_DIR, "fet-cl.tar.gz")
    subprocess.run(["curl", "-sL", "-o", tar_path,
        "https://github.com/nico-alvz/fet-mirror/releases/download/v7.10.5/fet-cl-7.10.5-linux-x86_64.tar.gz"],
        check=True)
    subprocess.run(["tar", "xzf", tar_path, "-C", BIN_DIR], check=True)
else:
    print("fet-cl ya está disponible en", FET_CL)

assert os.path.isfile(FET_CL), "No se encontró el binario fet-cl"
print("Entorno:", "Google Colab" if IN_COLAB else "Jupyter local")
print("Directorio de trabajo:", PROJECT_DIR)
print("fet-cl:", FET_CL)


In [ ]:
!pip install -q openpyxl pandas matplotlib


## 1. Cargar el archivo de Oferta Académica

Coloca tu archivo `.xlsx` de oferta académica en el mismo directorio que este notebook (en Colab, súbelo con
el panel de archivos, o simplemente ejecuta la siguiente celda: si no encuentra ningún `.xlsx`, te lo pedirá).


In [ ]:
import glob

def find_xlsx():
    return [f for f in glob.glob(os.path.join(PROJECT_DIR, "*.xlsx"))]

xlsx_files = find_xlsx()

if not xlsx_files and IN_COLAB:
    from google.colab import files
    print("No se encontró ningún archivo .xlsx en", PROJECT_DIR)
    print("Sube tu archivo de Oferta Académica (.xlsx):")
    uploaded = files.upload()
    xlsx_files = find_xlsx()

def pedir_archivo():
    """Binder/Jupyter: botón para subir el .xlsx (la oferta académica no vive en el repositorio)."""
    import ipywidgets as w
    from IPython.display import display
    up = w.FileUpload(accept=".xlsx", multiple=False, description="Subir .xlsx")
    def _guardar(cambio):
        v = up.value
        items = [(f["name"], f["content"]) for f in v] if not isinstance(v, dict) else [(k, d["content"]) for k, d in v.items()]
        for nom, contenido in items:
            with open(os.path.join(PROJECT_DIR, nom), "wb") as fh: fh.write(bytes(contenido))
            print("Guardado:", nom, "-> ahora vuelve a ejecutar esta celda")
    up.observe(_guardar, names="value"); display(up)

if not xlsx_files:
    print("La oferta académica NO está en el repositorio. Sube el .xlsx con el botón (o arrástralo al panel de archivos) y vuelve a ejecutar esta celda.")
    pedir_archivo()
    raise FileNotFoundError(
        f"No se encontró ningún archivo .xlsx de Oferta Académica en {PROJECT_DIR}. "
        "Coloca el archivo .xlsx junto a este notebook (o en /content si usas Colab) y vuelve a ejecutar esta celda."
    )

XLSX = xlsx_files[0]
print("Usando archivo:", XLSX)


In [ ]:
import openpyxl
import pandas as pd

wb = openpyxl.load_workbook(XLSX, data_only=True)
ws = wb.worksheets[0]

# La oferta académica trae un par de filas de encabezado/aviso antes de la fila de títulos de columna;
# se detecta automáticamente la primera fila que luce como encabezado (muchas celdas de texto no vacías).
all_rows = list(ws.iter_rows(values_only=True))
header_row_idx = 0
for i, row in enumerate(all_rows[:10]):
    non_empty = sum(1 for v in row if v not in (None, ""))
    if non_empty >= max(5, len(row) * 0.5):
        header_row_idx = i
        break

header = all_rows[header_row_idx]
df = pd.DataFrame(all_rows[header_row_idx + 1:], columns=header)
df = df.dropna(axis=1, how="all")
print(df.shape)
df.head()


## 2. Recortar a un subconjunto manejable

Para que la demostración sea legible y rápida de resolver, se trabaja con un **subconjunto** de la oferta
académica (por ejemplo, un semestre/área específico) en vez del archivo completo.


In [ ]:
AREA_COL = "Área"

if AREA_COL in df.columns and df[AREA_COL].notna().any():
    counts = df[AREA_COL].value_counts()
    candidates = counts[counts >= 20]
    chosen_area = candidates.idxmin() if len(candidates) else counts.idxmax()
    subset = df[df[AREA_COL] == chosen_area].copy().reset_index(drop=True)
    print(f"Subconjunto elegido: {AREA_COL} = {chosen_area!r} ({len(subset)} filas de {len(df)})")
else:
    subset = df.copy().reset_index(drop=True)
    print(f"No se encontró la columna '{AREA_COL}'; se usa la oferta académica completa ({len(subset)} filas).")

subset["Docente"] = (subset["AP_PATERNO_PROFESOR"].fillna("Sin Asignar") + " " + subset["NOMBRE_PROFESOR"].fillna("")).str.strip()
subset["GroupName"] = "G" + subset["MATERIA_CURSO"].astype(str).str.replace("-", "", regex=False) + "_" + subset["Seccion"].astype(str)
subset["Cupo"] = subset["Cupo"].fillna(30).astype(int)

subjects = subset[["MATERIA_CURSO", "Titulo"]].drop_duplicates().reset_index(drop=True)
teacher_subjects = subset.groupby("Docente")["MATERIA_CURSO"].unique()
groups = subset[["GroupName", "MATERIA_CURSO", "Cupo"]].drop_duplicates(subset=["GroupName"])
day_cols = [c for c in ["Lunes","Martes","Miercoles","Jueves","Viernes","Sabado","Domingo"] if c in subset.columns]
real_rooms = sorted(set(v for col in day_cols for v in subset[col].dropna().unique()))

print(f"{len(subjects)} asignaturas, {len(groups)} grupos/secciones, "
      f"{len(teacher_subjects)} docentes, {len(real_rooms)} salas reales")
subjects


## 3. Inventar la capacidad de las salas

La oferta académica trae los **códigos reales de sala** (p. ej. `R-044`) pero no su capacidad. Ese único dato
se inventa aquí, de forma plausible, para poder aplicar la restricción de espacio de FET.


In [ ]:
import random
random.seed(42)
room_capacity = {r: random.choice([35, 40, 45, 50, 60, 70]) for r in real_rooms}
pd.Series(room_capacity, name="Capacidad (inventada)").sort_index()


## 4. Construir el archivo `.fet` (modelo de entrada de FET)

Se genera el XML nativo de FET con: días/bloques horarios (tomados de los bloques reales del archivo),
asignaturas, docentes (con las asignaturas que dictan realmente), salas (código real + capacidad inventada),
un año con un grupo por sección, y una actividad por cada sesión semanal real (fila) de cada sección — sin
fijar día, bloque ni sala: eso lo debe resolver FET.

Además de `ConstraintBasicCompulsorySpace` (que evita que dos actividades compartan sala a la misma hora), se
agrega para cada asignatura una `ConstraintSubjectPreferredRooms` listando todas las salas reales como candidatas
válidas: sin esto FET trata la sala como "sin especificar" y no la asigna, aunque la solución de horario sea
correcta.


In [ ]:
from xml.sax.saxutils import escape as x

days = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday"]
block_labels = sorted(subset["Bloque"].dropna().unique()) if "Bloque" in subset.columns else []
if block_labels:
    times = subset.dropna(subset=["Bloque"]).drop_duplicates("Bloque").set_index("Bloque")
    hours = [f"{b} ({times.loc[b, 'Inicio']}-{times.loc[b, 'Fin']})" if "Inicio" in times.columns else str(b)
             for b in block_labels]
else:
    hours = [f"Periodo {i+1}" for i in range(8)]

agid_map = {g: i + 1 for i, g in enumerate(groups["GroupName"])}
activities_meta = []

parts = []
parts.append('<?xml version="1.0" encoding="UTF-8"?>\n<fet version="7.10.5">\n')
parts.append("<Mode>Official</Mode>\n")
parts.append("<Institution_Name>Demo FET - Oferta Academica</Institution_Name>\n")
parts.append("<Comments>Generado a partir de un archivo de Oferta Academica</Comments>\n")

parts.append(f"<Days_List>\n<Number_of_Days>{len(days)}</Number_of_Days>\n")
for d in days:
    parts.append(f"<Day>\n\t<Name>{x(d)}</Name>\n</Day>\n")
parts.append("</Days_List>\n")

parts.append(f"<Hours_List>\n<Number_of_Hours>{len(hours)}</Number_of_Hours>\n")
for h in hours:
    parts.append(f"<Hour>\n\t<Name>{x(h)}</Name>\n</Hour>\n")
parts.append("</Hours_List>\n")

parts.append("<Subjects_List>\n")
for _, r in subjects.iterrows():
    parts.append(f'<Subject>\n\t<Name>{x(r["MATERIA_CURSO"])}</Name>\n\t<Comments>{x(r["Titulo"])}</Comments>\n</Subject>\n')
parts.append("</Subjects_List>\n")

parts.append("<Activity_Tags_List>\n</Activity_Tags_List>\n")
parts.append("<Buildings_List>\n</Buildings_List>\n")

parts.append("<Teachers_List>\n")
for t in sorted(teacher_subjects.index):
    parts.append("<Teacher>\n")
    parts.append(f"\t<Name>{x(t)}</Name>\n\t<Target_Number_of_Hours>0</Target_Number_of_Hours>\n\t<Qualified_Subjects>\n")
    for subj in teacher_subjects[t]:
        parts.append(f"\t\t<Qualified_Subject>{x(subj)}</Qualified_Subject>\n")
    parts.append("\t</Qualified_Subjects>\n\t<Comments></Comments>\n</Teacher>\n")
parts.append("</Teachers_List>\n")

parts.append("<Rooms_List>\n")
for rn in real_rooms:
    parts.append(f"<Room>\n\t<Name>{x(rn)}</Name>\n\t<Building></Building>\n\t<Capacity>{room_capacity[rn]}</Capacity>\n\t<Virtual>false</Virtual>\n\t<Comments>Capacidad inventada</Comments>\n</Room>\n")
parts.append("</Rooms_List>\n")

parts.append('<Students_List>\n<Year>\n\t<Name>Cohorte demo</Name>\n\t<Number_of_Students>0</Number_of_Students>\n\t<Comments></Comments>\n\t<Number_of_Categories>0</Number_of_Categories>\n\t<Separator>_</Separator>\n')
for _, r in groups.iterrows():
    parts.append(f'\t<Group>\n\t\t<Name>{x(r["GroupName"])}</Name>\n\t\t<Number_of_Students>{int(r["Cupo"])}</Number_of_Students>\n\t\t<Comments></Comments>\n\t</Group>\n')
parts.append("</Year>\n</Students_List>\n")

parts.append("<Activities_List>\n")
activity_id = 1
for _, r in subset.iterrows():
    parts.append("<Activity>\n")
    parts.append(f'\t<Teacher>{x(r["Docente"])}</Teacher>\n')
    parts.append(f'\t<Subject>{x(r["MATERIA_CURSO"])}</Subject>\n')
    parts.append(f'\t<Students>{x(r["GroupName"])}</Students>\n')
    parts.append("\t<Duration>1</Duration>\n\t<Total_Duration>1</Total_Duration>\n")
    parts.append(f"\t<Id>{activity_id}</Id>\n")
    parts.append(f'\t<Activity_Group_Id>{agid_map[r["GroupName"]]}</Activity_Group_Id>\n')
    parts.append("\t<Active>true</Active>\n\t<Comments></Comments>\n</Activity>\n")
    activities_meta.append({
        "Id": activity_id, "GroupName": r["GroupName"], "MATERIA_CURSO": r["MATERIA_CURSO"],
        "Titulo": r["Titulo"], "Docente": r["Docente"], "Cupo": int(r["Cupo"]),
        "TipoOriginal": r["Tipo"] if "Tipo" in subset.columns else None,
    })
    activity_id += 1
parts.append("</Activities_List>\n")

parts.append("<Space_Constraints_List>\n<ConstraintBasicCompulsorySpace>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintBasicCompulsorySpace>\n")
# ConstraintBasicCompulsorySpace only penalizes room clashes/overcapacity — it does NOT by
# itself make FET search for a room. Each subject needs an explicit room preference list so
# the solver actually assigns one (unspecifiedPreferredRoom defaults to true otherwise).
for _, r in subjects.iterrows():
    subj = r["MATERIA_CURSO"]
    parts.append("<ConstraintSubjectPreferredRooms>\n\t<Weight_Percentage>100</Weight_Percentage>\n")
    parts.append(f"\t<Subject>{x(subj)}</Subject>\n")
    parts.append(f"\t<Number_of_Preferred_Rooms>{len(real_rooms)}</Number_of_Preferred_Rooms>\n")
    for rn in real_rooms:
        parts.append(f"\t<Preferred_Room>{x(rn)}</Preferred_Room>\n")
    parts.append("\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintSubjectPreferredRooms>\n")
parts.append("</Space_Constraints_List>\n")
parts.append("<Time_Constraints_List>\n<ConstraintBasicCompulsoryTime>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintBasicCompulsoryTime>\n</Time_Constraints_List>\n")
parts.append("<Timetable_Generation_Options_List>\n</Timetable_Generation_Options_List>\n")
parts.append("</fet>\n")

WORK_DIR = os.path.join(PROJECT_DIR, "fet_run")
os.makedirs(WORK_DIR, exist_ok=True)
FET_INPUT = os.path.join(WORK_DIR, "oferta_academica.fet")
with open(FET_INPUT, "w", encoding="utf-8") as f:
    f.write("".join(parts))

activities_meta_df = pd.DataFrame(activities_meta)
print(f"{len(activities_meta_df)} actividades (sesiones semanales) escritas en {FET_INPUT}")


## 5. Ejecutar FET (`fet-cl`) para generar el horario

In [ ]:
OUTPUT_DIR = os.path.join(WORK_DIR, "out")
env = dict(os.environ, QT_QPA_PLATFORM="offscreen")
result = subprocess.run(
    [FET_CL, f"--inputfile={FET_INPUT}", f"--outputdir={OUTPUT_DIR}", "--timelimitseconds=20"],
    capture_output=True, text=True, env=env,
)
print(result.stdout)
print(result.stderr)
assert "Generation successful" in result.stdout, "FET no logró generar el horario"


## 6. Leer el horario generado por FET y cruzarlo con los metadatos

In [ ]:
import xml.etree.ElementTree as ET

activities_xml = glob.glob(os.path.join(OUTPUT_DIR, "timetables", "*", "*_activities.xml"))[0]
tree = ET.parse(activities_xml)
sol_rows = []
for a in tree.getroot().findall("Activity"):
    sol_rows.append({
        "Id": int(a.find("Id").text),
        "Day": a.find("Day").text,
        "Hour": a.find("Hour").text,
        "Room": a.find("Room").text,
    })
solution = pd.DataFrame(sol_rows)
schedule = activities_meta_df.merge(solution, on="Id").sort_values(["Day", "Hour"])
schedule


## 7. Verificar que el horario generado es válido

Un horario válido no debe tener a un mismo **docente** ni a un mismo **grupo** en dos actividades a la vez.


In [ ]:
teacher_clashes = schedule.groupby(["Docente", "Day", "Hour"]).size()
group_clashes = schedule.groupby(["GroupName", "Day", "Hour"]).size()
room_clashes = schedule.dropna(subset=["Room"]).groupby(["Room", "Day", "Hour"]).size()

print("Choques de docente:", int((teacher_clashes > 1).sum()))
print("Choques de grupo:  ", int((group_clashes > 1).sum()))
print("Choques de sala:   ", int((room_clashes > 1).sum()))


## 8. Visualizar el horario de un grupo y de un docente

In [ ]:
import matplotlib.pyplot as plt

def plot_grid(df, key_col, key_value, title):
    sub = df[df[key_col] == key_value]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.set_xlim(0, len(days)); ax.set_ylim(0, len(hours))
    ax.set_xticks([i + 0.5 for i in range(len(days))]); ax.set_xticklabels(days)
    ax.set_yticks([i + 0.5 for i in range(len(hours))]); ax.set_yticklabels(hours[::-1])
    ax.grid(True, color="#ddd")
    for spine in ax.spines.values():
        spine.set_visible(False)
    for _, r in sub.iterrows():
        di = days.index(r["Day"]); hi = hours.index(r["Hour"])
        y = len(hours) - hi - 1
        label = f'{r["MATERIA_CURSO"]}' + (f'\n{r["Room"]}' if pd.notna(r["Room"]) else "")
        ax.add_patch(plt.Rectangle((di, y), 1, 1, color="#4C72B0", alpha=0.85))
        ax.text(di + 0.5, y + 0.5, label, ha="center", va="center", fontsize=8, color="white")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_grid(schedule, "GroupName", groups.iloc[0]["GroupName"], f'Horario generado por FET — {groups.iloc[0]["GroupName"]}')
top_teacher = schedule["Docente"].value_counts().idxmax()
plot_grid(schedule, "Docente", top_teacher, f"Horario generado por FET — Docente: {top_teacher}")


## 9. Conclusión

FET (motor de generación de horarios, https://lalescu.ro/liviu/fet/) tomó como entrada las asignaturas, docentes,
secciones y salas reales de un archivo de Oferta Académica, y **generó desde cero** una asignación de día/bloque
(y sala, cuando corresponde) para cada sesión semanal, sin choques de docente, grupo ni sala. Esto demuestra el
funcionamiento del motor de timetabling de FET operando sobre un caso realista.
